### Install dependencies for this tutorial

This tutorial requires Python 3.12, which Colab features when selecting runtime version 2026.07. Adjust your used runtime accordingly before running this tutorial. After installation, you might need to restart the runtime once.

In [ ]:
!git clone -b new_example https://github.com/andreasMazur/geoconv.git
%cd geoconv
!pip install -e ".[tensorflow]"

## Getting started with GeoConv

GeoConv is a library that provides classes and functions for deep learning on curved 2D surfaces. Implementing and training surface neural networks comes with two main tasks:
1. **Local surface charting**: in contrast to convolutions on images, which silently benefit from the fact that the 2D Euclidean domain allows for a **global** coordinate system along which the template of a CNN can be aligned, curved surfaces require **local surface charts** to be calculated before training.
2. **Neural network configuration**: similarly to graph neural networks which can be implemented differently, think of graph convolutions and graph attention networks for instance, different ways of implementing surface convolutions exist. GeoConv provides you with a variety of possible surface convolutions.

In this tutorial, we will go through the standard workflow when working with surface CNNs using bent MNIST images as an example. We go over the following sections:

1. Loading and bending MNIST images
2. Preprocessing the bent images
3. Training a surface neural network classifier

### Creating a grid for MNIST images

GeoConv's core preprocessing functions (i.e. charting algorithms and computation of so-called barycentric coordinates) typically expect triangle meshes as input.

Hence, to train on MNIST, we associate each MNIST image to a triangle mesh. Thereby, each vertex of the mesh carries the color of a pixel in the image.

In [ ]:
from geoconv_examples.mnist.preprocess import create_grid

# MNIST images have size 28 x 28.
N_MNIST_PIXELS = 28

# Create the grid: each MNIST pixel corresponds to one grid point.
# 'create_grid' further creates a Delaunay triangulation for the grid.
grid = create_grid(n_vertices=N_MNIST_PIXELS)

# Visualize the grid
print(f"N vertices: {grid.vertices.shape[0]}")
print(f"M faces: {grid.faces.shape[0]}")
grid.show(viewer="notebook", flags={"wireframe": True})

Now that we have the grid, we can combine the geometry with an MNIST image.

In [ ]:
import tensorflow_datasets as tfds

import os
import trimesh

# Deactivate TensorFlow logging for better readability
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Load MNIST images
mnist = tfds.load("mnist", split="train", shuffle_files=True, as_supervised=True)

# Take one element
image, label = next(iter(mnist))

# Show shape of image and label
print(f"Image shape: {image.numpy().shape}")
print(f"Label shape: {label.numpy().shape}")

# Visualize grid with image colors
grid.visual.vertex_colors = trimesh.visual.interpolate(
    image.numpy().reshape(-1),
    color_map="viridis"
)
grid.show()

When we work with flat surfaces, as represented by `grid` now, we can learn patterns in signals defined on top of them by simply using Euclidean 2D convolutions. However, these stop working once we bend the surface. For example, we can bend the plane by superimposing a Gaussian bell over the grid:

In [ ]:
import numpy as np


# Define center of the Gaussian bell
x0, y0, _ = grid.vertices.mean(axis=0)

# Define amplitude of the Gaussian bell and its stddev
sigma = 0.3
amplitude = 1.0

# Distance of every vertex from the Gaussian center
vertices = grid.vertices.copy()
dx = vertices[:, 0] - x0
dy = vertices[:, 1] - y0

# Gaussian value at every vertex
gaussian = amplitude * np.exp(-(dx**2 + dy**2) / (2 * sigma**2))

# Increase the existing z-coordinate
vertices[:, 2] += gaussian

# Create the new surface
gaussian_grid = trimesh.Trimesh(vertices=vertices, faces=grid.faces)
gaussian_grid.visual.vertex_colors = grid.visual.vertex_colors.copy()
gaussian_grid.show()

### Preprocess: surface charts, barycentric coordinates and GeoConv's 'Atlas'-class

We now have a non-Euclidean geometry that we can use as the domain in which we define MNIST images. The goal of this tutorial is to make a surface CNN learn the MNIST images from this non-Euclidean geometry. To this end, we need to process the geometry into a format that can be propagated into a surface CNN layer. When using GeoConv, a geometry is presented to a surface CNN layer in the form of so-called **barycentric coordinates**.

What are barycentric coordinates and why do they represent the geometry? The answer lies in how GeoConv implements surface convolutions.

Roughly speaking, GeoConv implements surface convolutions by positioning a polar-grid of points onto each vertex of the geometry, so-called **template vertices**. A template vertex most likely falls onto a triangle and not directly onto a mesh vertex. But only mesh vertices are associated to color values. Therefore, we interpolate the color values of surrounding mesh vertices at each template vertex using barycentric coordinates as interpolation coefficients. These coefficients are determined using **local surface charts** that represent local geodesic distance and angular directions in 2D polar coordinates. Eventually, these interpolations are used for convolution.

GeoConv offers the `Atlas` class, which summarizes preprocessing functions for both computing local surface charts and computing barycentric coordinates.

In [ ]:
from geoconv.preprocessing.atlas import Atlas


# Compute local surface charts by instantiating an Atlas over the geometry
chart_radius = 0.2
template_radius = chart_radius / 2
n_radial, n_angular = 2, 4
atlas = Atlas(
    triangle_mesh=gaussian_grid,  # The geometry for which charts are computed.
    max_radius=chart_radius,  # How large can the charts become? Value in between [0, 1] (after mesh normalization).
    method="fmm",  # charting algorithm, to be selected from ["fmm", "hdm", "dgpc", "tp"], used to compute local charts for subsequent bc computations
    normalization_method="hdm",  # charting algorithm used to compute the geodesic diameter of the geometry for mesh normalization
    processes=10  # number of concurrent processes for computing local charts and barycentric coordinates
)

# Using the compute charts, compute a set of barycentric coordinates
atlas.determine_barycentric_coordinates(
    n_radial=n_radial,  # The number of radial dots per direction
    n_angular=n_angular,  # The number of angular dots per radial level, i.e., concentric circle
    template_radius=chart_radius / 2  # The template radius determines the maximum geodesic distance of a template vertex to the convolution center
)

# Save the atlas
atlas.save("./gaussian_bell_atlas.hdf5")

We can now visualize the charts stored in the `Atlas` using its metho `visualize_chart`. Feel free to play with the `n_radial`, `n_angular`, `chart_radius`, `template_radius` and `method` parameters as you seem fit in the cell above.

In [ ]:
from geoconv.preprocessing.bc.bc_utils import create_template_matrix


atlas.visualize_chart(
    chart_idx=42,  # Choose the index of the chart origin. Change this value to visualize different charts.
    show_statistics=True,  # Adds statistics about the chart extension of all charts in the Atlas to the plot.
    additional_scatter_dots=create_template_matrix(  # This function is used by both the 'Atlas' class and the base-convolution class for any surface convolution to define template vertices.
        n_radial=n_radial,
        n_angular=n_angular,
        radius=template_radius,
        in_cart=True
    ).reshape(-1, 2)
)

### Preparing the dataset

Given that we now have an `Atlas`, we can prepare the dataset for surface CNN training. For each image, the dataset should additionally return barycentric coordinates:

In [ ]:
from geoconv.preprocessing.atlas import load_atlas

import tensorflow as tf


def dataset(set_type, batch_size, for_gauge_equiv_arch=True):
    # Read the MNIST images
    mnist_ds = tfds.load("mnist", split=set_type, shuffle_files=True, as_supervised=True)

    # Load the atlas that contains the surface charts and barycentric coordinates
    loaded_atlas = load_atlas("./gaussian_bell_atlas.hdf5")

    # Read the set of computed barycentric coordinates using triples '(n_radial, n_angular, template_radius)' as keys
    bc = loaded_atlas.barycentric_coordinates[(n_radial, n_angular, template_radius)]

    # A function that prepares each image of MNIST
    def combine_image_and_bc(mnist_image, mnist_label):
        # Normalize image
        mnist_image = tf.cast(mnist_image, tf.float32) / 255.

        # Reshape image:
        # the surface CNN interprets the n-th color value as the carried signal for the n-th mesh vertex
        mnist_image = tf.reshape(mnist_image, (784, 1))

        if for_gauge_equiv_arch:
            # 1.) Surface CNNs that implement so-called 'gauge-equivariant' surface convolutions require
            # parallel transportation angles, which GeoConv puts into the 3-dimension of the last axis
            # in the barycentric coordinates tensor.

            # 2.) Furthermore, these architectures process complex-valued features, requiring any input
            # to be even-dimensional. As MNIST comes with gray-scale pictures, we thus concat a zero
            # each gray-scale value, imitating a zero imaginary part of the complex feature.
            imaginary_values = tf.zeros((784, 1), dtype=tf.float32)
            mnist_image = tf.concat([mnist_image, imaginary_values], axis=-1)

            # With rotation angles for parallel transports
            return (mnist_image, bc), mnist_label
        else:
            # Remove the rotation angles for parallel transports when using non gauge-equivariant surface CNNs
            return (mnist_image, bc[..., :2]), mnist_label

    # Combine MNIST images with geometries
    mnist_ds = mnist_ds.map(combine_image_and_bc)

    # Return batched MNIST
    return mnist_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

The surface CNN thus receives the following tensor shapes during training:

In [ ]:
data = dataset(set_type="train", batch_size=32, for_gauge_equiv_arch=False)

for (mnist_image, bc), label in data.take(1):
    print(
        f"MNIST image shape: {tf.shape(mnist_image)}\n"
        f"Label shape: {tf.shape(label)}\n"
        f"Barycentric coordinates tensor shape: {tf.shape(bc)}"
    )

The MNIST images tensor contains a batch of 32 images, whereby each image is represented as a set of 784 scalars (more commonly, the last axis carries n dimensional vertex signals with n > 1).

The Labels tensor contains a label for each of the 32 images.

The barycentric coordinates tensor axes carry the following information:
1. **Batch dimension.** Each image is associated to its own barycentric coordinates tensor.
2. **The convolution center dimension.** The template is centered in each vertex of the input shape, thus in vertex-many locations.
3. **The radial dimension.** A template is configured to have 'n_radial' concentric circles.
4. **The angular dimension.** A template is configured to have 'n_angular' equidistant angular template vertices per concentric circle.
5. **The triangle dimension.** A template vertex is positioned in a triangle of the input geometry, whose vertices carry signals (in this example: color values) which are interpolated at the template vertex.
6. **The information dimension.** Contains two scalars when using non gauge-equivariant surface convolutions and three scalars when using gauge-equivariant surface convolutions.
    - The **first dimension** of this axis carries the barycentric coordinate, i.e., the interpolation coefficient for the currently considered triangle vertex.
    - The **second dimension** of this axis carries the vertex index to pick the signal from the image tensor that is associated to the currently cosidered triangle vertex.
    - The **third dimension** carries an angle that is used for the parallel transport of a neighboring signal to the convolution center vertex (not used in this example).

### Model definition

We have now prepared the dataset that we can use for training. In the following, we implement a **geodesic convolution neural network (GCNN)** that shall learn to predict MNIST images given the barycentric coordinates for the Gaussian bell defined above.

In [ ]:
from geoconv.tensorflow.layers import ConvGeodesic, AngularMaxPooling

# Define input layers (do not list batch dimension here)
image_input = tf.keras.Input(shape=(N_MNIST_PIXELS ** 2, 1), name="image_input", dtype=tf.float32)
bc_input = tf.keras.Input(shape=(N_MNIST_PIXELS ** 2, n_radial, n_angular, 3, 2), name="bc_input", dtype=tf.float32)

# Initialize variables for forward pass
signal = image_input

# Forward pass
for output_dim in [32, 32]:
    signal = ConvGeodesic(
        output_dim=output_dim,  # Defines the dimensionality of the output embedding for the layer
        template_radius=template_radius,  # The template radius should be set equal to the radius used for computing barycentric coordinates
        activation="relu",  # Defines the activation function
        rotation_delta=1  # Defines the rotation delta: a value that rotates the template of the surface convolution
    )([signal, bc_input])
    signal = AngularMaxPooling()(signal)

# Convert all local descriptors, i.e., vertex features, to one global image descriptor
signal = tf.keras.layers.GlobalMaxPool1D(data_format="channels_last")(signal)

# Classify the image descriptor
output = tf.keras.layers.Dense(10, activation="linear")(signal)

# Compile the model
surface_cnn = tf.keras.Model(inputs=[image_input, bc_input], outputs=output, name="mnist_model")
surface_cnn.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["sparse_categorical_accuracy"]
)

### Training

Now we have:
1. Preprocessed the geometry and inspected the result
2. Created the dataset that combines both MNIST images with the Gaussian bell
3. Defined our surface CNN

We can thus run the training as follows:

In [ ]:
import json


# Initialize training and test data
train_data = dataset(set_type="train", batch_size=64, for_gauge_equiv_arch=False)
test_data = dataset(set_type="test", batch_size=64, for_gauge_equiv_arch=False)

# Show model summary
surface_cnn.summary()

# Train model
history = surface_cnn.fit(x=train_data, batch_size=64, epochs=2, validation_data=test_data)

# Save model
surface_cnn.save("./surface_cnn.keras")

# Save history
with open(f"./train_history.json", "w") as f:
    json.dump(history.history, f, indent=4)

With that, you now know the basics of GeoConv. More detailed information is listed in the doc-strings. Furthermore, the 'geoconv_examples'-package features more complex surface learning examples.